# Neural IDS Training
This notebook trains the Neural IDS model, extracts metrics, and updates the comparison table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
import os

from src.models.neural_ids import NeuralIDS
from src.models.baselines import evaluate_binary


In [ ]:
PROJECT_ROOT = Path(os.getcwd()).parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
NEURAL_DIR = RESULTS_DIR / "neural_model"

NEURAL_DIR.mkdir(parents=True, exist_ok=True)

try:
    X_train = np.load(PROCESSED_DIR / "X_train.npy")
    X_test = np.load(PROCESSED_DIR / "X_test.npy")
    y_train = np.load(PROCESSED_DIR / "y_train.npy")
    y_test = np.load(PROCESSED_DIR / "y_test.npy")
    print(f"Data loaded: X_train {X_train.shape}, X_test {X_test.shape}")
except FileNotFoundError:
    print("Error: Processed data not found. Please run preprocessing pipeline first.")


In [ ]:
if "X_train" in locals():
    # Hyperparams
    epochs = 30
    batch_size = 1024
    lr = 1e-3
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = NeuralIDS(input_dim=X_train.shape[1]).to(device)
    
    # Class weighting
    n_pos = y_train.sum()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    
    train_losses = []
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb).squeeze(-1)
            loss = criterion(logits, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= len(X_train)
        train_losses.append(epoch_loss)
        
        if epoch % 5 == 0:
            print(f"Epoch {epoch}/{epochs} Loss: {epoch_loss:.4f}")
            
    # Save model weights
    torch.save(model.state_dict(), NEURAL_DIR / "neural_ids_weights.pth")
    
    # Plot and save training curves
    plt.plot(range(1, epochs + 1), train_losses)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Neural IDS Training Loss")
    plt.savefig(NEURAL_DIR / "training_loss.png")
    plt.show()
    
    pd.DataFrame({"epoch": range(1, epochs + 1), "loss": train_losses}).to_csv(NEURAL_DIR / "training_loss.csv", index=False)


In [ ]:
if "X_train" in locals():
    # Predict and Evaluate
    model.eval()
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_test_t).squeeze(-1)
        probs = torch.sigmoid(logits).cpu().numpy()
    
    y_pred = (probs >= 0.5).astype(int)
    y_proba = np.column_stack([1 - probs, probs])
    
    # Save per-sample predicted probabilities
    np.save(NEURAL_DIR / "test_probabilities.npy", probs)
    
    metrics = evaluate_binary(y_test, y_pred, y_proba, model_name="NeuralIDS")
    
    # Load previous baseline comparison and add this model
    comp_csv = RESULTS_DIR / "baseline_comparison.csv"
    if comp_csv.exists():
        df_summary = pd.read_csv(comp_csv)
        # Add if not already present
        if "NeuralIDS" not in df_summary["Model"].values:
            new_row = pd.DataFrame([{
                "Model": "NeuralIDS",
                "Accuracy": metrics["accuracy"],
                "Precision": metrics["precision_macro"],
                "Recall": metrics["recall_macro"],
                "F1": metrics["f1_macro"]
            }])
            df_summary = pd.concat([df_summary, new_row], ignore_index=True)
            df_summary.to_csv(comp_csv, index=False)
            df_summary.to_markdown(RESULTS_DIR / "baseline_comparison.md", index=False)
            print("Updated comparison table.")
        display(df_summary)
    else:
        print("Baseline comparison not found. Run 02_baseline.ipynb first.")
